In [22]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from eval_utils import get_vecser_for_file, vecser_similarity_matrix
from dotenv import load_dotenv
from tqdm import tqdm
from eval_utils import vecser_similarity_evidence_for_files

from langchain_ollama import ChatOllama
from langchain.tools import tool

from ollama import chat
from ollama import ChatResponse

# Load environment variables from .env file
load_dotenv()

True

In [23]:
# Initialize the ChatOllama model with the specified model name
model_name = 'qwen2.5-coder:7b'

# and initialize the ChatOllama instance
chat_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0.7
)

In [24]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [25]:
f1 = gjt_file_paths[0]
f2 = gjt_file_paths[1]
print(f1)
print(f2)

bars_string, graph_res, token_res, adapter_res = vecser_similarity_evidence_for_files(
    f1,
    f2,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    topk=10
)

/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Mean_To_Me.mxl
/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/My_One_And_Only_Love.mxl


In [26]:
print(bars_string)

Piece 1:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj6 D:min7 G:7 
bar 10: C:maj6 
bar 11: D:9 G:7 
bar 12: C:maj6 
bar 13: F:7 E:7 
bar 14: A:min7 
bar 15: F:7 E:7 

Piece 2:
bar 0: F:maj7 D:min7 
bar 1: G:min7 C:7 C#:dim 
bar 2: D:min7 A#:maj7 
bar 3: A:min7 D:7 
bar 4: G:min7 C:7 C#:dim 
bar 5: D:min7 G:7 
bar 6: G:min7 C:7 
bar 7: A:min7 D:7 G:min7 C:7 
bar 8: F:maj6 B:hdim7 E:7 
bar 9: A:min7 
bar 10: B:hdim7 E:7(b9) 
bar 11: A:min7 
bar 12: B:hdim7 E:7(b9) 
bar 13: A:min A:minmaj7 
bar 14: A:min7 D:7 
bar 15: G:min7 D:7 



In [27]:
print(graph_res)

Graph model evidence:
piece 1, bar 12: ['C:maj6'] | piece 2, bar 11: ['A:min7'] | 0.9996343851089478
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.7704873085021973
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.7364344596862793
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 14: ['A:min7', 'D:7'] | 0.7003163695335388
piece 1, bar 11: ['D:9', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6958389282226562
piece 1, bar 0: ['G:maj6', 'G#:dim7'] | piece 2, bar 10: ['B:hdim7', 'E:7(b9)'] | 0.6921544671058655
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6828776597976685
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6638666391372681
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6544992327690125
piece 1, bar 9: ['G:maj6', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6302633881568909



In [28]:
print(token_res)

Token model evidence:
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.7414132952690125
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7253557443618774
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6307624578475952
piece 1, bar 6: ['G:maj6', 'E:min7'] | piece 2, bar 8: ['F:maj6', 'B:hdim7', 'E:7'] | 0.5227781534194946
piece 1, bar 14: ['A:min7'] | piece 2, bar 4: ['G:min7', 'C:7', 'C#:dim'] | 0.5074173808097839
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.5039945840835571
piece 1, bar 12: ['C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.5002508163452148
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.494163453578949
piece 1, bar 14: ['A:min7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.4927017092704773
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 11: ['A:min7'] | 0.4927017092704773



In [29]:
print(adapter_res)

Adapter model evidence:
piece 1, bar 10: ['C:maj6'] | piece 2, bar 9: ['A:min7'] | 0.939810037612915
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.8129515647888184
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7870017290115356
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7619063258171082
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.7256702780723572
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 0: ['F:maj7', 'D:min7'] | 0.7003436088562012
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6731694936752319
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6675118803977966
piece 1, bar 3: ['C:maj7', 'F:9'] | piece 2, bar 0: ['F:maj7', 'D:min7'] | 0.6441857814788818
piece 1, bar 11: ['D:9', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6413013935089111



In [30]:
central_prompt = '''
You are a music harmony expert and you will be give the chord sequences of two pieces, per bar.
You job is to provide an account of the similarities between the two harmonies.\n\n
'''

tool_prompt = '''
You can use the assistance of a model that assessed the following similarities (maximum 1, minimum -1)
between bars of the two pieces:
'''

In [31]:
print(central_prompt + bars_string)


You are a music harmony expert and you will be give the chord sequences of two pieces, per bar.
You job is to provide an account of the similarities between the two harmonies.


Piece 1:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj6 D:min7 G:7 
bar 10: C:maj6 
bar 11: D:9 G:7 
bar 12: C:maj6 
bar 13: F:7 E:7 
bar 14: A:min7 
bar 15: F:7 E:7 

Piece 2:
bar 0: F:maj7 D:min7 
bar 1: G:min7 C:7 C#:dim 
bar 2: D:min7 A#:maj7 
bar 3: A:min7 D:7 
bar 4: G:min7 C:7 C#:dim 
bar 5: D:min7 G:7 
bar 6: G:min7 C:7 
bar 7: A:min7 D:7 G:min7 C:7 
bar 8: F:maj6 B:hdim7 E:7 
bar 9: A:min7 
bar 10: B:hdim7 E:7(b9) 
bar 11: A:min7 
bar 12: B:hdim7 E:7(b9) 
bar 13: A:min A:minmaj7 
bar 14: A:min7 D:7 
bar 15: G:min7 D:7 



In [32]:
response: ChatResponse = chat(
  model='qwen2.5-coder:7b',
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string,
    }
  ],
  keep_alive=0
)
basic_response = response['message']['content']
print(basic_response)

Both Piece 1 and Piece 2 exhibit a variety of harmonies that create interesting and complex musical textures. Let's analyze the similarities between the two pieces:

### Key Elements:
1. **Chord Progressions**: Both pieces utilize a combination of major, minor, and diminished chords. This indicates a rich harmonic language where tensions are created and then resolved.

2. **Modulation and Chromaticism**:
   - **Modulation**: Both pieces show instances where the key modulates to different keys (e.g., from G major in Piece 1 to F major in Piece 2). This adds variety and complexity to the overall harmonic landscape.
   - **Chromaticism**: The use of chromatic chords (diminished seventh, half-diminished seventh) is evident in both pieces. These chords introduce color and movement within the harmony.

3. **Harmonic Density**:
   - Both pieces pack a significant amount of harmonic information into each bar, with multiple chord changes that create a dynamic and engaging listening experience.


In [33]:
response: ChatResponse = chat(
  model='qwen2.5-coder:7b',
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + token_res,
    }
  ],
  keep_alive=0
)
token_response = response['message']['content']
print(token_response)

### Similarities Between Piece 1 and Piece 2

#### Key Chords:
Both pieces utilize a variety of major and minor chords in their harmonic progressions, showcasing a rich tapestry of tonal colors. 

- **Major Chords**: 
  - Both piece 1 and piece 2 frequently incorporate major chords such as `G:maj7`, `C:maj7`, and `D:9`. These major chords often function as dominant chords that provide tension and resolution within their progressions.
  
- **Minor Chords**:
  - Minor chords like `A:min7` and `D:min7` are prominent in both pieces. The minor mode typically evokes a sense of sadness, melancholy, or introspection, making these chords versatile for expressing different emotional contexts.

#### Dominant Seventh Chords (V7):
Both pieces extensively use dominant seventh chords (`G:maj7`, `A:min7`, `D:7`) as pivotal points in their harmonic progressions. The dominant seventh chord is a cornerstone of Western harmony and serves to create tension that is ultimately resolved by a tonic chord.

###

In [34]:
response: ChatResponse = chat(
  model='qwen2.5-coder:7b',
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + graph_res,
    }
  ],
  keep_alive=0
)
graph_response = response['message']['content']
print(graph_response)

Both pieces showcase a variety of harmonic and melodic elements that give them their distinct sound, but there are several similarities between the two harmonies:

### Key Chords and Voicings:
- **Common Major V chords**: Both piece 1 and piece 2 contain major vii° (diminished seventh) chords. For example, piece 1 has G#dim7 in bar 0 and Dmin7 in bar 6, while piece 2 has Fmaj7 in bar 0 and D:min7 in bar 5.
- **V ii chords**: Both pieces make use of the V ii chord progression, where a major dominant (V) chord is followed by a minor subdominant (ii). For instance, piece 1 uses Amin7 in bar 1, following G:maj7 from bar 0, while piece 2 has G:min7 in bar 5 following D:min7 from bar 4.
- **Dominant and Subdominant Relationships**: Both pieces leverage dominant (V) and subdominant (ii) chords, forming a strong sense of tension and resolution. For example, piece 1 uses Amin7 in bar 3 following C:maj7 from bar 2, while piece 2 has A:min7 in bar 5 after D:min7.

### Minor Chords and Modalities:

In [35]:
response: ChatResponse = chat(
  model='qwen2.5-coder:7b',
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + adapter_res,
    }
  ],
  keep_alive=0
)
adapter_response = response['message']['content']
print(adapter_response)

**Similarities Between the Two Harmonies**

The two pieces share several structural similarities and common chord progressions. Here's an overview of the key parallels:

### Common Chords and Progressions

1. **Progression: I - ii - vi - I**
   - Piece 1: 
     - Bar 0: G:maj6 (I)
     - Bar 2: G:maj7, D:min7, G:7 (ii - vi - I)
   - Piece 2:
     - Bar 0: F:maj7 (I)
     - Bar 2: D:min7, A#:maj7 (ii - vi)

2. **Progression: ii - V - ii**
   - Both pieces use this progression at some point.
   - Piece 1:
     - Bar 3: C:maj7, F:9 (ii - V)
   - Piece 2:
     - Bar 3: A:min7, D:7 (ii - V)

### Common Minor Seventh Chords

Both pieces frequently utilize minor seventh chords, providing a rich and expressive harmonic texture.
- **Piece 1:** A:min7, G:maj6, E:7
- **Piece 2:** G:min7, F:maj7, C#:dim7

### Similar Bar-by-Bar Comparisons

The adapter model evidence indicates several bars with high similarity:
1. **Bar 10**: Both pieces feature a major chord (C:maj6 and A:maj6) that stand out.
  